Debug Teacher Circuits and Predictions

In [1]:
import sys
from pathlib import Path
root = Path.cwd()
if (root / 'qcgpt').exists():
    sys.path.insert(0, str(root))
elif (root.parent / 'qcgpt').exists():
    sys.path.insert(0, str(root.parent))


In [2]:
import numpy as np
import torch
from qcgpt.data.dataset import SimplifiedCircuitDataset
from qcgpt.data.qiskit_utils import sample_random_qiskit_circuit, simplify_qiskit_circuit
from qcgpt.simulators.qiskit_sim import qiskit_to_circuit
from qcgpt.encoding import circuit_to_tokens, tokens_to_circuit
from qcgpt.evaluation.visualize import format_circuit
from qcgpt.data.specs import build_spec_sequence_batch
from qcgpt.evaluation.metrics import quantum_fidelity_from_spec, gate_count
from qcgpt.models.policy import CircuitPolicy
from qcgpt.gates import VOCAB, PAD_ID, BOS_CIRC_ID, EOS_CIRC_ID


Inspect Qiskit Simplified Gates

In [3]:
for i in range(5):
    qc_raw = sample_random_qiskit_circuit(n_qubits=2, max_depth=8)
    qc_simp = simplify_qiskit_circuit(qc_raw)
    names = [inst.name for inst, _, _ in qc_simp.data]
    print(f'Example {i+1} simplified gate names:', names)
    try:
        circ = qiskit_to_circuit(qc_simp)
        print(format_circuit(circ))
    except Exception as e:
        print('Conversion failed:', str(e))
    print('-'*40)


Example 1 simplified gate names: ['cz', 'h', 'cx', 'cz', 's', 'h']
00: CZ q0 q1
01: H q1
02: CX q0 q1
03: CZ q1 q0
04: S q0
05: H q1
----------------------------------------
Example 2 simplified gate names: ['y', 'h', 'cz']
00: Y q0
01: H q1
02: CZ q0 q1
----------------------------------------
Example 3 simplified gate names: ['s', 'y']
00: S q0
01: Y q1
----------------------------------------
Example 4 simplified gate names: ['t']
00: T q0
----------------------------------------
Example 5 simplified gate names: ['y', 's', 'y', 't', 'y', 'h']
00: Y q0
01: S q0
02: Y q1
03: T q1
04: Y q1
05: H q1
----------------------------------------


C:\Users\CaioV\AppData\Local\Temp\ipykernel_36308\255175647.py:4: DeprecationWarning: Treating CircuitInstruction as an iterable is deprecated legacy behavior since Qiskit 1.2, and will be removed in Qiskit 2.0. Instead, use the `operation`, `qubits` and `clbits` named attributes.
  names = [inst.name for inst, _, _ in qc_simp.data]
C:\Users\CaioV\AppData\Local\Temp\ipykernel_36308\255175647.py:4: DeprecationWarning: Treating CircuitInstruction as an iterable is deprecated legacy behavior since Qiskit 1.2, and will be removed in Qiskit 2.0. Instead, use the `operation`, `qubits` and `clbits` named attributes.
  names = [inst.name for inst, _, _ in qc_simp.data]
C:\Users\CaioV\AppData\Local\Temp\ipykernel_36308\255175647.py:4: DeprecationWarning: Treating CircuitInstruction as an iterable is deprecated legacy behavior since Qiskit 1.2, and will be removed in Qiskit 2.0. Instead, use the `operation`, `qubits` and `clbits` named attributes.
  names = [inst.name for inst, _, _ in qc_simp.d

Inspect Dataset Teacher Circuits

In [ ]:
ds = SimplifiedCircuitDataset(num_samples=10, n_qubits=2, raw_max_depth=8)
items = [ds[i] for i in range(5)]
for idx, item in enumerate(items):
    circ_teacher = tokens_to_circuit(item['circ_tokens'].tolist())
    print(f'Teacher {idx+1}  Gates={gate_count(circ_teacher)}  Tokens={len(item["circ_tokens"])}')
    print(format_circuit(circ_teacher))
    print('-'*40)


Model Predictions vs Teachers

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CircuitPolicy(vocab_size=len(VOCAB)).to(device)
ckpt = Path('checkpoints/supervised_epoch_005.pt')
if ckpt.exists():
    state = torch.load(str(ckpt), map_location=device)
    model.load_state_dict(state['model_state_dict'])
model.eval()
for idx, item in enumerate(items):
    spec_tensor = item['spec_tensor'].numpy()
    spec_batch_np, spec_pad_mask_np = build_spec_sequence_batch([spec_tensor])
    spec_b = torch.tensor(spec_batch_np, dtype=torch.float32, device=device)
    pad_b = torch.tensor(spec_pad_mask_np, dtype=torch.bool, device=device)
    with torch.no_grad():
        seqs, logp = model.sample_circuit_tokens(spec_b, pad_b, BOS_CIRC_ID, EOS_CIRC_ID, max_len=64)
    seq = [t for t in seqs[0].tolist() if t != PAD_ID]
    circ_pred = tokens_to_circuit(seq)
    print(f'Prediction {idx+1}  Gates={gate_count(circ_pred)}')
    print(format_circuit(circ_pred))
    fid = quantum_fidelity_from_spec(spec_tensor, circ_pred)
    print(f'Fidelity: {fid:.3f}')
    print('-'*40)
